In [28]:
import requests
import tiktoken
import torch
from torch import nn
import datasets


tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [29]:
engine_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")

with open("model.py", "w") as f:
    f.write(engine_res.text)
    # print(engine_res.text)
print("Downloaded model.py")

# Downloading train_utils.py
train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")

with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
    # print(engine_res.text)
print("Downloaded train_utils.py")

data_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")

with open("data.py", "w") as f:
    f.write(data_res.text)
    # print(engine_res.text)
print("Downloaded data.py")

#Downloading custom_gpt_v2.pth weights
wt_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain_v2.pth")
with open("tara_n1_pretrain_v2.pth", "wb") as f:
    f.write(wt_res.content)
print("Downloaded modle weights")


Downloaded model.py
Downloaded train_utils.py
Downloaded data.py
Downloaded modle weights


In [30]:
from model import *
from train_utils import *
from data import *


config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=4,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

# Mid Trainign with `SimpleStories/SimpleStories`, `pszemraj/simple_wikipedia_LM`, and  `rojagtap/bookcorpus` datasets

## Download and prep Dataset


In [31]:
from datasets import load_dataset

simple_stories_ds = load_dataset("SimpleStories/SimpleStories")
simple_wiki_ds = load_dataset("pszemraj/simple_wikipedia_LM")
bookcorpus_ds = load_dataset("rojagtap/bookcorpus")


In [32]:
print(len(simple_stories_ds))
print(len(simple_wiki_ds))
print(len(bookcorpus_ds))

2
3
1


In [33]:
bookcorpus_ds["train"][0:10]

{'text': ['the half-ling book one in the fall of igneeria series kaylee soderburg copyright 2013 kaylee soderburg all rights reserved .',
  'isbn : 1492913731 isbn-13 : 978-1492913733 for my family , who encouraged me to never stop fighting for my dreams chapter 1 summer vacations supposed to be fun , right ?',
  'i wish i had a better answer to that question .',
  'starlings , new york is not the place youd expect much to happen .',
  'its a small quiet town , the kind where everyone knows your name .',
  'its a place where your parents wouldnt even care if you stayed out late biking with your friends .',
  'only because everyone felt so safe , so comfy .',
  'they dont know the half of it .',
  'but i do .',
  'i know it all and starlings is not the place where you want to be after dark .']}

In [34]:
import numpy as np
import random

simple_stories_examples = []
simple_wiki_examples = []
bookcorpus_examples = []

for sample in simple_stories_ds["train"]:
    simple_stories_examples.append(sample["story"])
    if len(simple_stories_examples) == 45000:
        break
print("Collected Simple Stories Examples")

for sample in simple_wiki_ds["train"]:
    simple_wiki_examples.append("# Title: "+sample["title"]+"\n"+sample["text"])
    if len(simple_wiki_examples) == 30000:
        break
print("Collected Simple Wiki Examples")

for i in range(0, len(bookcorpus_ds["train"]), 10):

    sample = [x for x in bookcorpus_ds["train"][i:i+10]["text"]]

    bookcorpus_examples.append("\n".join(sample))

    if len(bookcorpus_examples) == 25000:
        break
print("Collected BookCoupus Examples")


print("Creating interleaved data")

choices = (["a"]*len(simple_stories_examples)) + (["b"]*len(simple_wiki_examples)) + (["c"]*len(bookcorpus_examples))

random.shuffle(choices)

data = []

for choice in choices:
    if choice == "a":
        data.append(simple_stories_examples.pop(0))
    elif choice == "b":
        data.append(simple_wiki_examples.pop(0))
    else:
        data.append(bookcorpus_examples.pop(0))

print("Interleaved dataset created")





Collected Simple Stories Examples
Collected Simple Wiki Examples
Collected BookCoupus Examples
Creating interleaved data
Interleaved dataset created


In [35]:
from tqdm.auto import tqdm
idx = 0
eot = tokenizer.eot_token
# pbar = tqdm(total = len(data), )
total_tokens = 0
for text in tqdm(data, desc="Calculating token counts"):
    # Tokenize + 1 for the <|endoftext|> delimiter
    tokens = tokenizer.encode(text, allowed_special="all")
    total_tokens += len(tokens) + 1

print(f"Total tokens: {total_tokens:,}")

mmap = np.lib.format.open_memmap("mid_training.npy", dtype=np.uint16, mode="w+", shape=(total_tokens, ))
for i in tqdm(range(1, len(data)+1), desc="Creating tokens"):
    text = data[i-1]
    tokens = tokenizer.encode(text, allowed_special="all")
    tokens.append(eot)

    length = len(tokens)
    mmap[idx: idx+length] = np.array(tokens, dtype=np.uint16)
    idx += length

mmap.flush()
print("Successfullly saved tokens.")

Calculating token counts:   0%|          | 0/100000 [00:00<?, ?it/s]

Total tokens: 24,512,087


Creating tokens:   0%|          | 0/100000 [00:00<?, ?it/s]

Successfullly saved tokens.


### Creating dataloaders

In [36]:
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import numpy as np
class MidTrainDataset(Dataset):
    def __init__(self, path, block_size):
        self.data = np.load(path, mmap_mode="r", allow_pickle=True)
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]

        return (
            torch.from_numpy(x.astype(np.int64)),
            torch.from_numpy(y.astype(np.int64))
        )


def create_data_doc(data):
    doc = ""
    pbar = tqdm(total=len(data))
    for sample in data:
        text = sample["story"]
        doc += f"{text}\n{tokenizer.decode([tokenizer.eot_token])}\n\n"
        pbar.update(1)


    return doc.strip()

In [37]:
full_dataset = MidTrainDataset("mid_training.npy", config.block_size)

train_split = int(0.9*len(full_dataset))

train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_split))
test_dataset = torch.utils.data.Subset(full_dataset, range(train_split, len(full_dataset)))

train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)

print(len(train_dataset))
print(len(test_dataset))

22059956
2451107


In [38]:
print(len(test_dataset))

2451107


## Training loop

In [39]:
model = CustomGPT(config)
state_dict = torch.load("tara_n1_pretrain_v2.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

In [40]:
from tqdm.auto import tqdm

steps = 2962
test_count = 500
test_steps = 50

train_iter = iter(train_dataloader)

avg_train_loss = 0

for step in tqdm(range(1, steps+1)):
    model.train()
    if not train_iter:
        train_iter = iter(train_dataloader)

    train_loss = train_step(model, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    avg_train_loss += train_loss
    if step%test_count == 0:
        avg_train_loss /= test_count

        test_loss = test_step(model, test_dataloader, test_steps, loss_fn, device)

        print(f"Step: {step} | Train loss: {avg_train_loss} | Test loss: {test_loss}")
        avg_train_loss = 0


  0%|          | 0/2962 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Step: 500 | Train loss: 3.559974455833435 | Test loss: 3.423905596733093


  0%|          | 0/50 [00:00<?, ?it/s]

Step: 1000 | Train loss: 3.28324267244339 | Test loss: 3.351139874458313


...

Step: 1500 | Train loss: 3.2185723571777345 | Test loss: 3.2165729427337646


  0%|          | 0/50 [00:00<?, ?it/s]

Step: 2000 | Train loss: 3.178012397289276 | Test loss: 3.1988970041275024


  0%|          | 0/50 [00:00<?, ?it/s]

Step: 2500 | Train loss: 3.1042331302165986 | Test loss: 3.142672734260559


In [41]:
model.eval()

query = "Once upon a time, there live a king "
context = torch.tensor(tokenizer.encode(query), dtype=torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    generated = model.generate(context)

print(tokenizer.decode(generated[0].tolist()).split("<|endoftext|>")[0].replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">"))


Once upon a time, there live a king ivee who dreams forever. The silliest figure with a broken pick shot explodes? With a big grin, the cloak fell from the warm earth as the river's edge grew darker and pulled up a hole above the ground. Gazing in the sky, he felt invisible. 

The sun with flowers danced across the earth, and the giant jumped out. Kim was running to the throne, ready to chase the land in return. In an instant, a boy named Leo appeared, saving the night from darkness. She wished he could never remember her. The best statue could be about love. Eddie was out, but in this amazing day, her heart charged with the great impact.


In [42]:
torch.save(model.state_dict(), "tara_n1_pretrain_v3.pth")